O trabalho tem como objetivo realizar uma análise de sentimentos em avaliações de produtos utilizando técnicas de Processamento de Linguagem Natural (PLN). A proposta busca identificar padrões de satisfação e insatisfação na categoria deeletronicos dos consumidores a partir de comentários realizados em plataformas de e-commerce, permitindo compreender melhor o comportamento do consumidor e os fatores que influenciam suas percepções sobre produtos e serviços.

Para o desenvolvimento do projeto, foi utilizada a base de dados “Brazilian E-Commerce Public Dataset by Olist”, disponível na plataforma Kaggle. A escolha dessa base ocorreu por ela conter avaliações reais de consumidores brasileiros, incluindo notas atribuídas aos produtos e comentários textuais das compras realizadas. Além disso, a base apresenta uma grande quantidade de dados, tornando mais adequada para aplicações de análise textual e mineração de dados.

A partir dessas informações, Inicialmente foram aplicadas etapas de pré-processamento textual com o objetivo de preparar os comentários para análises posteriores de sentimentos. Essas etapas incluem limpeza dos dados, remoção de registros vazios, padronização textual e eliminação de ruídos presentes nos comentários.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# bibliotecas
import pandas as pd
import re
import nltk

In [5]:
# dataset
reviews = pd.read_csv('/content/drive/MyDrive/trabalho1/olist_order_reviews_dataset.csv')

items = pd.read_csv('/content/drive/MyDrive/trabalho1/olist_order_items_dataset.csv')

products = pd.read_csv('/content/drive/MyDrive/trabalho1/olist_products_dataset.csv')

translation = pd.read_csv('/content/drive/MyDrive/trabalho1/product_category_name_translation.csv')

In [6]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [7]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [9]:
translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [11]:
# fazendo joins dos csvs e vendo quail categoria tem mais avaliações:
reviews_items = reviews.merge(
    items[['order_id', 'product_id']],
    on='order_id',
    how='inner'
)

dados = reviews_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='inner'
)
dados['product_category_name'].value_counts().head(15)

,count
product_category_name,
cama_mesa_banho,11137
beleza_saude,9645
esporte_lazer,8640
moveis_decoracao,8331
informatica_acessorios,7849
utilidades_domesticas,6943
relogios_presentes,5950
telefonia,4517
ferramentas_jardim,4329


In [13]:
eletronicos = dados[
    dados['product_category_name'] == 'eletronicos'
].copy()
eletronicos.shape

(2749, 9)

In [16]:
eletronicos['review_comment_message'].isnull().sum()

np.int64(0)

In [17]:
eletronicos = eletronicos.dropna(subset=['review_comment_message'])

In [18]:
eletronicos.shape

(1169, 9)

In [19]:
# selecionando apenas as colunas necessárias para a análise
eletronicos = eletronicos[['review_score', 'review_comment_message']]

eletronicos.head()

,review_score,review_comment_message
38,5,A compra foi realizada facilmente.\r\nA entreg...
66,1,recebi somente 1 controle Midea Split ESTILO.\...
67,1,recebi somente 1 controle Midea Split ESTILO.\...
311,5,muito bom e rapido quando vi ja tinha chegado
312,5,muito bom e rapido quando vi ja tinha chegado


In [23]:
# padronizando letras para minusculo:
eletronicos['review_comment_message'] = (
    eletronicos['review_comment_message']
    .str.lower()
)

# removendo pontuação:
eletronicos['review_comment_message'] = (
    eletronicos['review_comment_message']
    .apply(lambda x: re.sub(r'[^\w\s]', '', x))
)

# removendo quebras de linha:
eletronicos['review_comment_message'] = (
    eletronicos['review_comment_message']
    .str.replace(r'\r\n', ' ', regex=True)
)

In [24]:
# removendo stopwords da base de dados para melhorar a analise:
# baixando stopwords:
import nltk
nltk.download('stopwords')

#importando:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('portuguese'))

# função:
def remover_stopwords(texto):
    palavras = texto.split()
    palavras_filtradas = [p for p in palavras if p not in stop_words]
    return ' '.join(palavras_filtradas)

# aplicando na base:
eletronicos['review_comment_message'] = (
    eletronicos['review_comment_message']
    .apply(remover_stopwords)
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [25]:
eletronicos.head()

,review_score,review_comment_message
38,5,compra realizada facilmente entrega efetuada a...
66,1,recebi somente 1 controle midea split estilo f...
67,1,recebi somente 1 controle midea split estilo f...
311,5,bom rapido vi ja chegado
312,5,bom rapido vi ja chegado


In [26]:
eletronicos.duplicated().sum()

np.int64(196)

In [28]:
# removendo registros duplicados
eletronicos = eletronicos.drop_duplicates()

# conferindo a base
eletronicos.shape

(973, 2)

In [29]:
eletronicos.duplicated().sum()

np.int64(0)

In [30]:
eletronicos.head()

,review_score,review_comment_message
38,5,compra realizada facilmente entrega efetuada a...
66,1,recebi somente 1 controle midea split estilo f...
311,5,bom rapido vi ja chegado
547,5,atendimento excelente
673,5,otima venda tudo perfeito
